<a href="https://colab.research.google.com/github/the-menna-sherif/Refreshing_ML_Muscle_Memory/blob/main/PySpark_ML_UseCase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic ML Prediction use case

Using pyspark and other libraries:

Following the second half of the tutorial below, we will use the features age and experience to predict salary.

Reference:
https://www.youtube.com/watch?v=_C8kWso4ne4

In [3]:
# create a spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Missing').getOrCreate()

In [4]:
# read data with out dtypes and col names
training_df = spark.read.csv('test1.csv', inferSchema=True, header=True)

In [5]:
training_df.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [6]:
training_df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [7]:
training_df.columns

['Name', 'age', 'Experience', 'Salary']

In pyspark, the way to do it is to group all the **independent** features using a **VectorAssembler** object.

['age', 'Experience'] ----> new feature using VectorAssembler ----> independent feature

CAN be done on str features after pre-proc (encoding. etc.)

In [8]:
from pyspark.ml.feature import VectorAssembler

featureassembler = VectorAssembler(inputCols=['age', 'Experience'], # cols used to predict
                                   outputCol="Independent Features")

In [9]:
output = featureassembler.transform(training_df) # combine columns to have 1:1 (feature vector:label)

In [10]:
output.show() # NEW FEATURE !

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent Features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [11]:
output.columns

['Name', 'age', 'Experience', 'Salary', 'Independent Features']

In [12]:
finalized_df = output.select('Independent Features', 'Salary') # NEW DF with only useful data for my model

In [13]:
finalized_df.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



# ML portion

1- Train Test Split -- here using **randomSplit**

2- Instantiate Regressor (i/p is featuures and target)

3- Train model

4- Predict

In [14]:
from pyspark.ml.regression import LinearRegression

## train test split
train_data, test_data = finalized_df.randomSplit([0.75,0.25])

## instantiate our regressor with the features & label
regressor = LinearRegression(featuresCol='Independent Features', labelCol='Salary')

## train data
regressor = regressor.fit(train_data)

**regressor.coefficients**

These values represent the direction and magnitude of featureXlabel relationships. These are the weights.
- positive coefficients indicate a direct relationship
- negative coefficients indicate an inverse one
- they come in a DenseVector of # of items = # features
- trainable params ♻

In [15]:
## this will show the learned
regressor.coefficients

DenseVector([-188.2353, 1564.7059])

**regressor.intercept**

The constant term (beta) in a linear regression model. Indicates the expected value of the dependent variable when all independent variables are zero.
This is the bias.

In [17]:
regressor.intercept

19200.0

In [18]:
pred_results = regressor.evaluate(test_data)

In [19]:
pred_results.predictions.show()

+--------------------+------+------------------+
|Independent Features|Salary|        prediction|
+--------------------+------+------------------+
|          [21.0,1.0]| 15000|16811.764705882335|
|          [24.0,3.0]| 20000|19376.470588235294|
|          [30.0,8.0]| 25000|26070.588235294163|
|         [31.0,10.0]| 30000| 29011.76470588242|
+--------------------+------+------------------+



pred_results has many more feature values from our predictions!

In [20]:
pred_results.meanAbsoluteError, pred_results.meanSquaredError

(1123.5294117646963, 1448512.1107266187)